In [1]:
# CARGAR ENTORNO DE TRABAJO
import sys
from pathlib import Path
import polars as pl

# notebook/jupyter → proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scr.python.clase_datalake import(ParquetStore)

In [ ]:
# Lectura dataframe particular
out_dir = "../../data/rawparquet"
amb = "madrid"
dom = "euros_pais"
estado = 0
ano = 2025
m = 10

store = ParquetStore(out_dir)

df = store.read(
    ambito=amb,
    dominio=dom,
    estado=estado,
    anio=ano,
    mes=m
)

df = df.to_pandas()

In [3]:
# Lectura de múltiples años y meses

out_dir = "../../data/rawparquet"
amb = "madrid"
dom = "euros_pais"

ano_ini = 2024
ano_fin = 2025
ano_def = 2023
meses = [9, 10, 11, 12]  

store = ParquetStore(out_dir)

df_completo = store.merge_parquet_range(
    ambito = amb,
    dominio = dom,
    anio_inicio = ano_ini,
    anio_definitivo = ano_def,  
    anio_fin = ano_fin,          
    meses=None,             
    lazy=False              
)

print(f"\n📊 Total registros: {len(df_completo):,}")
print(f"📅 Periodo: {df_completo['anio'].min()}-{df_completo['mes'].min():02d} a {df_completo['anio'].max()}-{df_completo['mes'].max():02d}")
print(f"\nColumnas: {df_completo.columns}")

# Convertir a Pandas para Data Wrangler
df_analysis = df_completo.to_pandas()
print(f"\n✅ DataFrame guardado como 'df_analysis' (Pandas)")
print(f"   Listo para Data Wrangler")

2026-01-16 10:21:21 - INFO - Agregando madrid/euros_pais 2024-2025 (def hasta 2023)
2026-01-16 10:21:21 - INFO - ✓ 22 parquets añadidos a la agregación



📊 Total registros: 8,474
📅 Periodo: 2024-01 a 2025-12

Columnas: ['flujo', 'pais', 'euros', 'estado', 'anio', 'mes']

✅ DataFrame guardado como 'df_analysis' (Pandas)
   Listo para Data Wrangler


In [7]:
# Lectura de múltiples años y meses
out_dir = "../../data/rawparquet"
amb = "madrid"
dom = "euros_sectores"
ano_ini = 1995
ano_fin = 2025
ano_def = 2023
meses = [9, 10, 11, 12]

# Definir filtros (None = sin filtro)
filtro_pais = [0, 1, 400] 
filtro_cod_taric = [0] 
filtro_cod_sector_economico = None  
filtro_flujo = [1]  

# Cargar datos en modo lazy
store = ParquetStore(out_dir)
df_lazy = store.merge_parquet_range(
    ambito=amb,
    dominio=dom,
    anio_inicio=ano_ini,
    anio_definitivo=ano_def,
    anio_fin=ano_fin,
    meses=meses,  
    lazy=True 
)

# Detectar columnas disponibles
columnas_disponibles = df_lazy.collect_schema().names()
print(f"Columnas disponibles en '{dom}':")
print(columnas_disponibles)
print()

# Aplicar filtros solo si la columna existe y el filtro no es None
df_lazy_filtered = df_lazy  # Crear copia para filtrar
filtros_aplicados = []

if filtro_flujo is not None and "flujo" in columnas_disponibles:
    df_lazy_filtered = df_lazy_filtered.filter(pl.col("flujo").is_in(filtro_flujo))
    filtros_aplicados.append(f"flujo: {filtro_flujo}")

if filtro_pais is not None and "pais" in columnas_disponibles:
    df_lazy_filtered = df_lazy_filtered.filter(pl.col("pais").is_in(filtro_pais))
    filtros_aplicados.append(f"pais: {filtro_pais}")

if filtro_cod_taric is not None and "cod_taric" in columnas_disponibles:
    df_lazy_filtered = df_lazy_filtered.filter(pl.col("cod_taric").is_in(filtro_cod_taric))
    filtros_aplicados.append(f"cod_taric: {filtro_cod_taric}")

if filtro_cod_sector_economico is not None and "cod_sector_economico" in columnas_disponibles:
    df_lazy_filtered = df_lazy_filtered.filter(pl.col("cod_sector_economico").is_in(filtro_cod_sector_economico))
    filtros_aplicados.append(f"cod_sector_economico: {filtro_cod_sector_economico}")

# Mostrar filtros aplicados
print("Filtros aplicados:")
if filtros_aplicados:
    for filtro in filtros_aplicados:
        print(f"  ✓ {filtro}")
else:
    print("  ⊘ Ninguno (sin filtros)")

# Ejecutar queries y materializar resultados
print("\nEjecutando queries...")
df_filtered = df_lazy_filtered.collect()

# Convertir a Pandas para Data Wrangler
df_analysis_filtered = df_filtered.to_pandas()

# Resumen
print(f"\n{'='*60}")
print(f"📊 RESUMEN")
print(f"{'='*60}")
print(f"Registros filtrados:  {len(df_analysis_filtered):,}")


print(f"\n✅ DataFrames disponibles para Data Wrangler:")
print(f"   • df_analysis_filtered → Datos filtrados ({len(df_analysis_filtered):,} registros)")

# Vista previa
print(f"\n🔍 Primeras 10 filas del DataFrame filtrado:")
display(df_analysis_filtered.head(10))

2026-01-16 10:08:28 - INFO - Agregando madrid/euros_sectores 1995-2025 (def hasta 2023)
2026-01-16 10:08:28 - INFO - ✓ 122 parquets añadidos a la agregación


Columnas disponibles en 'euros_sectores':
['flujo', 'nivel_sector_economico', 'cod_sector_economico', 'euros', 'estado', 'anio', 'mes']

Filtros aplicados:
  ✓ flujo: [1]

Ejecutando queries...

📊 RESUMEN
Registros filtrados:  23,034

✅ DataFrames disponibles para Data Wrangler:
   • df_analysis_filtered → Datos filtrados (23,034 registros)

🔍 Primeras 10 filas del DataFrame filtrado:


,flujo,nivel_sector_economico,cod_sector_economico,euros,estado,anio,mes
0,1,2,22,1.173584e+07,1,1995,9
1,1,4,4310,6.681995e+06,1,1995,9
2,1,3,1B0,1.803000e+01,1,1995,9
3,1,4,8150,1.353845e+06,1,1995,9
4,1,4,1800,1.723063e+06,1,1995,9
5,1,2,73,5.020663e+06,1,1995,9
6,1,4,8200,8.504198e+05,1,1995,9
7,1,4,7300,5.020663e+06,1,1995,9
8,1,2,41,3.849723e+06,1,1995,9
9,1,3,431,6.681995e+06,1,1995,9
